In [0]:
from pyspark.sql import functions as F, Window
from pyspark.ml.feature import VectorAssembler

In [0]:
# Read from recovered base table (Jun 2025 – Feb 2026, 36 devices, 6.4M rows)
# Already includes: 800W filter, fault forward-fill, delta_seconds
df = spark.table("teams.sensorx.df_sx_800_with_delta")
df = df.drop("serialNumber", "payload_fault_faultName", "GeneratorType")

print(f"Rows: {df.count():,}")
print(f"Devices: {df.select('properties_deviceId').distinct().count()}")
print(f"Date range: {df.selectExpr('MIN(timeStamp)').first()[0]} to {df.selectExpr('MAX(timeStamp)').first()[0]}")
print(f"Columns: {df.columns}")

In [0]:
# Explicit sensor columns
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_onTime",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]

df = df.na.drop(subset=sensor_cols)

# Encode properties_deviceId as integer index
device_ids = df.select("properties_deviceId").distinct().orderBy("properties_deviceId")
device_ids = device_ids.withColumn("deviceId_index", (F.dense_rank().over(Window.orderBy("properties_deviceId")) - 1).cast("double"))
df_encoded = df.join(device_ids, on="properties_deviceId", how="inner")

# Horizon (binary: failure within H_days or not)
H_days = 15
H_seconds = H_days * 86400

w_horizon = (
    Window.partitionBy("properties_deviceId")
    .orderBy(F.col("timeStamp").cast("long"))
    .rangeBetween(0, H_seconds)
)

df_encoded = df_encoded.withColumn(
    "failure_horizon",
    F.max(F.col("payload_fault_state").cast("int")).over(w_horizon)
)

# Lag features (n_lags = 20)
n_lags = 20
w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

lag_exprs = [F.lag(col_name, L).over(w).alias(f"{col_name}{L}")
             for L in range(1, n_lags + 1) for col_name in sensor_cols]
df_encoded = df_encoded.select("*", *lag_exprs)

lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

# Break lineage via Delta temp path (memory-efficient, no named table)
temp_path = "/tmp/sensorx_lag_prep"
df_encoded.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(temp_path)
df_encoded = spark.read.format("delta").load(temp_path)

# Deviation features: 24-hour time-based rolling average
w_daily = (
    Window.partitionBy("properties_deviceId")
    .orderBy(F.col("timeStamp").cast("long"))
    .rangeBetween(-86400, 0)
)

dev_sensors = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
]
dev_cols = []
dev_exprs = []

for col_name in dev_sensors:
    avg_col = f"{col_name}_avg_daily"
    dev_col = f"{col_name}_dev_daily"
    dev_cols.extend([avg_col, dev_col])
    dev_exprs.append(F.avg(col_name).over(w_daily).alias(avg_col))
    dev_exprs.append((F.col(col_name) - F.avg(col_name).over(w_daily)).alias(dev_col))

df_encoded = df_encoded.select("*", *dev_exprs)

print(f"Added {len(dev_cols)} deviation features (24-hour window)")

# Drop nulls and assemble
df_clean = df_encoded.na.drop(subset=lag_cols + dev_cols)

feature_input_cols = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
df_features = assembler.transform(df_clean)

df_features = df_features.select("timeStamp", "features", F.col("failure_horizon").alias("label"))

# Cache to avoid recomputation during multiple actions below
df_features = df_features.cache()
row_count = df_features.count()
print(f"\nTotal rows after feature engineering: {row_count:,}")

if row_count > 0:
    print(f"Feature vector size: {df_features.select('features').first()[0].size}")
    print(f"  - {len(sensor_cols)} sensor cols")
    print(f"  - {len(lag_cols)} lag cols")
    print(f"  - {len(dev_cols)} deviation cols (24-hour window)")
    print(f"  - 1 deviceId index")
    print(f"\nLabel distribution (H={H_days} days):")
    df_features.groupBy("label").count().orderBy("label").show()
else:
    print("WARNING: No rows survived feature engineering! Check upstream data.")

# Chronological train/test split 80/20
cutoff_epoch = df_features.selectExpr(
    "percentile_approx(CAST(timeStamp AS BIGINT), 0.8)"
).first()[0]

train = df_features.filter(F.col("timeStamp").cast("bigint") <= cutoff_epoch)
test  = df_features.filter(F.col("timeStamp").cast("bigint") >  cutoff_epoch)

print(f"\nTrain: {train.count():,} rows")
print(f"Test:  {test.count():,} rows")
print(f"\nTrain label distribution:")
train.groupBy("label").count().orderBy("label").show()
print(f"Test label distribution:")
test.groupBy("label").count().orderBy("label").show()

In [0]:
train.write.mode("overwrite").saveAsTable("teams.sensorx.train_data")
test.write.mode("overwrite").saveAsTable("teams.sensorx.test_data")